In [ ]:
import os, glob, math, random
import numpy as np
from PIL import Image
import hashlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# 0) Reproducibility
# ============================================================
SEED = 614
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🔥 DEVICE:", DEVICE)

# ============================================================
# 1) 경로 및 로컬 캐싱 (LNO 스타일)
# ============================================================
PROJECT_PATH = '/content/drive/MyDrive/super_solutioner/LNO_base'
DRIVE_DATA_PATH = os.path.join(PROJECT_PATH, 'data/DIV2K')
LOCAL_DATA_PATH = '/content/DIV2K'
os.makedirs(LOCAL_DATA_PATH, exist_ok=True)

def setup_div2k_from_drive(split="valid"):
    zip_name = f"DIV2K_{split}_HR.zip"
    drive_zip_path = os.path.join(DRIVE_DATA_PATH, zip_name)
    extracted_folder = os.path.join(LOCAL_DATA_PATH, f"DIV2K_{split}_HR")

    if not os.path.exists(drive_zip_path):
        print(f"❌ 에러: {drive_zip_path} 파일이 없습니다!")
        return

    if not os.path.exists(extracted_folder):
        print(f"[{split}] 🚀 드라이브에서 코랩 로컬로 압축 해제 중...")
        os.system(f"unzip -q {drive_zip_path} -d {LOCAL_DATA_PATH}")
        print(f"[{split}] ✅ 완료!")
    else:
        print(f"[{split}] ⚡ 로컬에 이미 준비됨.")

setup_div2k_from_drive("valid")

# ============================================================
# 2) 실험 경로 세팅 (CoDA-LNO 버전)
# ============================================================
EXP_DIRS = {
    # 방금 학습하신 업그레이드 버전 체크포인트 경로
    "coda_lno": os.path.join(PROJECT_PATH, 'checkpoints_codalno_upgrade'),
    # (선택) 비교를 위해 이전 베이스라인 경로가 있다면 추가하세요
    # "baseline": os.path.join(PROJECT_PATH, 'checkpoints_codalno_base'),
}

DIV2K_ROOT = LOCAL_DATA_PATH
VALID_GLOB = os.path.join(DIV2K_ROOT, "DIV2K_valid_HR", "*.png")
valid_files = sorted(glob.glob(VALID_GLOB))
assert len(valid_files) == 100, f"Expected 100 images, got {len(valid_files)}"
print("✅ Valid images:", len(valid_files))

In [ ]:
# ============================================================
# 3) Utils (same as srno_exp_base_final.py / srno_comparison.py)
# ============================================================
def norm_01_to_m11(x):
    return (x - 0.5) / 0.5

def denorm_m11_to_01(x):
    return x * 0.5 + 0.5

def make_coord_grid(h, w, device, batch_size=1):
    y = torch.linspace(-1, 1, h, device=device)
    x = torch.linspace(-1, 1, w, device=device)
    yy, xx = torch.meshgrid(y, x, indexing="ij")
    coords = torch.stack([xx, yy], dim=-1).view(1, -1, 2)  # (1,N,2)
    cell = torch.tensor([2.0/h, 2.0/w], device=device, dtype=torch.float32).view(1, 2)
    if batch_size != 1:
        coords = coords.repeat(batch_size, 1, 1)
        cell = cell.repeat(batch_size, 1)
    return coords, cell

def rgb_to_y(img):  # img (...,3) in [0,1]
    return 0.2567 * img[...,0] + 0.5041 * img[...,1] + 0.0979 * img[...,2] + 16/255

@torch.no_grad()
def calc_psnr_y(pred01, target01, shave=2):
    pred01 = pred01.clamp(0,1)
    target01 = target01.clamp(0,1)

    pred01 = pred01[..., shave:-shave, shave:-shave]
    target01 = target01[..., shave:-shave, shave:-shave]

    pred = pred01.permute(0,2,3,1)
    tgt  = target01.permute(0,2,3,1)

    y_pred = rgb_to_y(pred)
    y_tgt  = rgb_to_y(tgt)

    mse = torch.mean((y_pred - y_tgt) ** 2).item()
    if not math.isfinite(mse) or mse <= 0:
        return float("nan") if not math.isfinite(mse) else 100.0
    return 20.0 * math.log10(1.0 / math.sqrt(mse))


# deterministic crop (file마다 항상 같은 좌표)
def deterministic_crop_xy(img_w, img_h, crop, key: str, seed=0):
    if img_w <= crop or img_h <= crop:
        return 0, 0
    s = f"{key}|{seed}".encode("utf-8")
    h = int(hashlib.md5(s).hexdigest(), 16)
    x = h % (img_w - crop + 1)
    y = (h // 1000003) % (img_h - crop + 1)
    return int(x), int(y)

def crop_pil(img, x, y, crop):
    return img.crop((x, y, x + crop, y + crop))

In [ ]:
# ============================================================
# 4. Model Definitions (LNO Baseline & CoDA-LNO)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ------------------------------------------------------------
# [A] LNO Baseline (DeepLNO_SR) 관련 클래스
# ------------------------------------------------------------
class PR2d_SR(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super(PR2d_SR, self).__init__()
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = (1 / (in_channels * out_channels))

        self.weights_pole1 = nn.Parameter(self.scale * torch.randn(in_channels, out_channels, self.modes1, dtype=torch.cfloat))
        self.weights_pole2 = nn.Parameter(self.scale * torch.randn(in_channels, out_channels, self.modes2, dtype=torch.cfloat))
        self.weights_residue = nn.Parameter(self.scale * torch.randn(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat))

    def output_PR(self, lambda1, lambda2, alpha, weights_pole1, weights_pole2, weights_residue):
        term1 = lambda1 - weights_pole1
        term2 = lambda2 - weights_pole2
        denom = term1.unsqueeze(-1) * term2.unsqueeze(-2)

        epsilon = 1e-5
        denom = torch.where(torch.abs(denom) < epsilon, denom + epsilon, denom)

        H = torch.div(weights_residue, denom)
        out_residue = torch.einsum("bixy,ioxy->boxy", alpha, H)
        return out_residue

    def forward(self, x, target_size):
        B, C, H, W = x.shape
        alpha = torch.fft.fft2(x, dim=[-2, -1])
        alpha_modes = alpha[..., :self.modes1, :self.modes2]

        omega1 = torch.fft.fftfreq(self.modes1, d=1/self.modes1).to(x.device) * 2 * np.pi * 1j
        omega2 = torch.fft.fftfreq(self.modes2, d=1/self.modes2).to(x.device) * 2 * np.pi * 1j
        lambda1 = omega1.reshape(1, 1, self.modes1)
        lambda2 = omega2.reshape(1, 1, self.modes2)

        out_res = self.output_PR(lambda1, lambda2, alpha_modes,
                                 self.weights_pole1, self.weights_pole2, self.weights_residue)

        x_out = torch.fft.ifft2(out_res, s=target_size, dim=[-2, -1])
        return torch.real(x_out)

class LNOBlock(nn.Module):
    def __init__(self, width, modes1, modes2):
        super().__init__()
        self.norm1 = nn.InstanceNorm2d(width)
        self.lno = PR2d_SR(width, width, modes1, modes2)
        self.act = nn.GELU()

        self.norm2 = nn.InstanceNorm2d(width)
        self.mlp = nn.Sequential(
            nn.Conv2d(width, width * 2, 1),
            nn.GELU(),
            nn.Conv2d(width * 2, width, 1)
        )

    def forward(self, x):
        target_size = (x.shape[2], x.shape[3])
        resid = self.lno(self.norm1(x), target_size)
        x = x + self.act(resid)

        resid = self.mlp(self.norm2(x))
        x = x + resid
        return x

class DeepLNO_SR(nn.Module):
    def __init__(self, in_channels=3, width=64, layers=4, modes1=32, modes2=32):
        super(DeepLNO_SR, self).__init__()
        self.lifting = nn.Conv2d(in_channels, width, 1)
        self.layers = nn.ModuleList([
            LNOBlock(width, modes1, modes2) for _ in range(layers)
        ])
        self.upsampler = PR2d_SR(width, width, modes1, modes2)
        self.proj = nn.Conv2d(width, in_channels, 1)

    def forward(self, x, target_size=None, scale_factor=None):
        # (호환성을 위해 scale_factor 인자 추가: 내부적으로 안 써도 에러 방지용)
        if target_size is None:
            target_size = (x.shape[2]*4, x.shape[3]*4)

        x_feat = self.lifting(x)
        for layer in self.layers:
            x_feat = layer(x_feat)
        x_up = self.upsampler(x_feat, target_size=target_size)
        out = self.proj(x_up)
        base = F.interpolate(x, size=target_size, mode='bicubic', align_corners=False)
        return out + base

# ------------------------------------------------------------
# [B] CoDA-LNO (CoDALNO_SR) 관련 클래스
# ------------------------------------------------------------
class LaplaceSpatialMixer(nn.Module):
    def __init__(self, dim, modes1=16, modes2=16):
        super().__init__()
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = (1 / (dim * dim))

        self.weights_pole1 = nn.Parameter(self.scale * torch.rand(1, 1, modes1, dtype=torch.cfloat))
        self.weights_pole2 = nn.Parameter(self.scale * torch.rand(1, 1, modes2, dtype=torch.cfloat))
        self.weights_residue = nn.Parameter(self.scale * torch.rand(dim, dim, modes1, modes2, dtype=torch.cfloat))

    def forward(self, x, target_size=None):
        B, C, H, W = x.shape
        if target_size is None: target_size = (H, W)

        alpha = torch.fft.fft2(x, dim=[-2, -1])
        alpha_modes = alpha[..., :self.modes1, :self.modes2]

        omega1 = torch.fft.fftfreq(self.modes1, d=1/self.modes1).to(x.device) * 2 * np.pi * 1j
        omega2 = torch.fft.fftfreq(self.modes2, d=1/self.modes2).to(x.device) * 2 * np.pi * 1j
        lambda1 = omega1.reshape(1, 1, self.modes1)
        lambda2 = omega2.reshape(1, 1, self.modes2)

        term1 = lambda1 - self.weights_pole1
        term2 = lambda2 - self.weights_pole2
        denom = term1.unsqueeze(-1) * term2.unsqueeze(-2)

        H_s = torch.div(self.weights_residue, denom)
        out_freq = torch.einsum("bixy,ioxy->boxy", alpha_modes, H_s)
        out = torch.fft.ifft2(out_freq, s=target_size, dim=[-2, -1])
        return torch.real(out)

class ChannelMixer(nn.Module):
    def __init__(self, dim, expansion=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim * expansion, 1),
            nn.GELU(),
            nn.Conv2d(dim * expansion, dim, 1)
        )
    def forward(self, x):
        return self.net(x)

class CoDALNOBlock(nn.Module):
    def __init__(self, dim, modes=16, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.InstanceNorm2d(dim)
        self.spatial_mixer = LaplaceSpatialMixer(dim, modes1=modes, modes2=modes)
        self.norm2 = nn.InstanceNorm2d(dim)
        self.channel_mixer = ChannelMixer(dim, expansion=mlp_ratio)
        self.alpha = nn.Parameter(torch.ones(1, dim, 1, 1))
        self.beta = nn.Parameter(torch.ones(1, dim, 1, 1))

    def forward(self, x):
        resid = self.spatial_mixer(self.norm1(x))
        x = x + self.alpha * resid
        resid = self.channel_mixer(self.norm2(x))
        x = x + self.beta * resid
        return x

class CoDALNO_SR(nn.Module):
    def __init__(self, in_channels=3, width=64, blocks=4, modes=16):
        super().__init__()
        self.width = width
        self.lifting = nn.Conv2d(in_channels, width, 1)

        self.scale_embed = nn.Sequential(
            nn.Linear(1, width),
            nn.GELU(),
            nn.Linear(width, width)
        )

        self.blocks = nn.ModuleList([
            CoDALNOBlock(width, modes=modes) for _ in range(blocks)
        ])

        self.upsampler = LaplaceSpatialMixer(width, modes1=modes, modes2=modes)
        self.proj = nn.Conv2d(width, in_channels, 1)

    def forward(self, x, target_size, scale_factor):
        if isinstance(scale_factor, (int, float)):
            scale_factor = torch.full((x.shape[0], 1), scale_factor, dtype=torch.float32, device=x.device)
        elif scale_factor.dim() == 1:
            scale_factor = scale_factor.unsqueeze(1)

        x_feat = self.lifting(x)
        s_emb = self.scale_embed(scale_factor).view(-1, self.width, 1, 1)
        x_feat = x_feat + s_emb

        shortcut = x_feat
        for block in self.blocks:
            x_feat = block(x_feat)
        x_feat = x_feat + shortcut

        x_up = self.upsampler(x_feat, target_size=target_size)
        out = self.proj(x_up)
        base = F.interpolate(x, size=target_size, mode='bicubic', align_corners=False)
        return out + base

In [ ]:
# ============================================================
# 5) 모델 로드 (LNO vs CoDA-LNO vs Upgrade 3종 세트 비교)
# ============================================================
import os

PROJECT_PATH = '/content/drive/MyDrive/super_solutioner/LNO_base'

EXP_DIRS = {
    "lno_base": os.path.join(PROJECT_PATH, "checkpoints", "epoch_2000.pth"),
    "codalno":  os.path.join(PROJECT_PATH, "checkpoints_codalno", "codalno_epoch_2000.pth"), # 여기서 에러 안 나게 세팅!
    "upgrade":  os.path.join(PROJECT_PATH, "checkpoints_codalno_upgrade", "codalno_latest.pth")
}

def load_eval_model(name, ckpt_path):
    if not os.path.exists(ckpt_path):
        print(f"⚠️ 파일 없음: {ckpt_path}")
        return None

    checkpoint = torch.load(ckpt_path, map_location="cpu")

    # ----------------------------------------------------
    # 🔥 핵심: 각 모델별 학습 당시의 파라미터(width, layers/blocks) 명시
    # ----------------------------------------------------
    if name == "lno_base":
        # DeepLNO 모델: width=64, layers=4
        model = DeepLNO_SR(in_channels=3, width=64, layers=4, modes1=32, modes2=32).to(DEVICE)

    elif name == "codalno":
        # 첫 번째 CoDA-LNO 모델: width=64, blocks=6
        model = CoDALNO_SR(in_channels=3, width=64, blocks=6, modes=32).to(DEVICE)

    elif name == "upgrade":
        # 업그레이드된 CoDA-LNO 모델: width=128, blocks=10
        model = CoDALNO_SR(in_channels=3, width=128, blocks=10, modes=32).to(DEVICE)

    # ----------------------------------------------------
    # State dict 불러오기
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    else:
        model.load_state_dict(checkpoint, strict=True)

    model.eval()
    return model

models = {}
for name, ckpt_path in EXP_DIRS.items():
    print(f"🔄 로드 중: {name:10s} <- {os.path.basename(ckpt_path)}")
    m = load_eval_model(name, ckpt_path)
    if m is not None:
        models[name] = m

print("✅ 준비된 모델:", list(models.keys()))

In [ ]:
# ============================================================
# 6) Eval one HR patch for a given scale (SRNO & LNO 호환 평가 모드)
# ============================================================
@torch.no_grad()
def eval_one_patch_psnr(hr01, model, *, scale, model_type="lno"):
    hr_size = hr01.shape[-1]
    lr_size = int(round(hr_size / float(scale)))
    lr_size = max(4, lr_size)

    # 파이토치 기반 Bicubic 통일 (입력 문제지 통일)
    lr01 = F.interpolate(hr01, size=(lr_size, lr_size), mode="bicubic", align_corners=False)

    if model_type == "srno":
        # 기존 SRNO 모델을 위한 로직
        hr_in = norm_01_to_m11(hr01)
        lr_in = norm_01_to_m11(lr01)
        coords, cell = make_coord_grid(hr_size, hr_size, DEVICE, batch_size=1)
        cell = cell * torch.tensor(scale, device=DEVICE, dtype=torch.float32).view(1, 1)

        pred = model(lr_in.to(DEVICE), coords, cell)
        pred = pred.view(1, hr_size, hr_size, 3).permute(0,3,1,2).contiguous()
        pred01 = denorm_m11_to_01(pred)

    elif model_type == "lno":
        # 🔥 LNO & CoDA-LNO 모델을 위한 로직 (에러 해결!)
        target_size = (hr_size, hr_size)
        # coords, cell 대신 target_size와 scale_factor 전달
        pred01 = model(lr01.to(DEVICE), target_size=target_size, scale_factor=float(scale))
        pred01 = torch.clamp(pred01, 0.0, 1.0)

    # 채점 기준 통일: 가장자리 2픽셀(shave=2) 무시하고 계산
    psnr = calc_psnr_y(pred01, hr01.to(DEVICE), shave=2)
    return psnr
# 기존 메모리에 살아있는 함수
HR_PATCH = 192

def load_hr_patch(path, hr_patch=192):
    img = Image.open(path).convert("RGB")
    W, H = img.size
    # 모든 모델이 똑같은 위치를 자르도록 해시(Hash) 기반의 고정 좌표 생성
    x, y = deterministic_crop_xy(W, H, hr_patch, key=os.path.basename(path), seed=SEED)
    hr = transforms.ToTensor()(crop_pil(img, x, y, hr_patch)).unsqueeze(0)  # (1,3,H,H) in [0,1]
    return hr, (x, y)
# ============================================================
# 7) Main: 평가 실행 루프 (고정 배율 & 랜덤 배율)
# ============================================================
def compute_avg_psnr(scales=(2.0, 3.0, 4.0)):
    results = {s: {name: [] for name in models.keys()} for s in scales}

    for path in tqdm(valid_files, desc="valid100"):
        hr01, _ = load_hr_patch(path, hr_patch=192)
        hr01 = hr01.to(DEVICE)

        for s in scales:
            for name, model in models.items():
                # 이름에 'lno'나 'upgrade'가 들어가면 lno 모드, 아니면 srno 모드
                m_type = "lno" if "lno" in name.lower() or "upgrade" in name.lower() else "srno"

                ps = eval_one_patch_psnr(hr01, model, scale=float(s), model_type=m_type)
                results[s][name].append(ps)

    summary = {}
    for s in scales:
        summary[s] = {}
        for name, vals in results[s].items():
            arr = np.array(vals, dtype=np.float64)
            summary[s][name] = {
                "mean": float(np.nanmean(arr)),
                "std":  float(np.nanstd(arr)),
            }
    return summary

def compute_random_scale_avg_psnr(scale_choices=(2.0, 3.0, 4.0)):
    results = {name: [] for name in models.keys()}

    for path in tqdm(valid_files, desc="valid100 (random-scale)"):
        hr01, _ = load_hr_patch(path, hr_patch=192)
        hr01 = hr01.to(DEVICE)

        s = float(random.choice(scale_choices))

        for name, model in models.items():
            m_type = "lno" if "lno" in name.lower() or "upgrade" in name.lower() else "srno"
            ps = eval_one_patch_psnr(hr01, model, scale=s, model_type=m_type)
            results[name].append(ps)

    summary = {}
    for name, vals in results.items():
        arr = np.array(vals, dtype=np.float64)
        summary[name] = {
            "mean": float(np.nanmean(arr)),
            "std":  float(np.nanstd(arr)),
        }
    return summary

In [ ]:
summary = compute_avg_psnr(scales=(2.0, 3.0, 4.0))

In [ ]:
# ============================================================
# 출력 1: 고정 스케일 평균 PSNR
# ============================================================
print("\n===== AVG Y-PSNR over DIV2K valid(100), HR_PATCH=192 =====")
for s in [2.0, 3.0, 4.0]:
    print(f"\n[scale x{s}]")
    # 하드코딩 대신 summary에 있는 키(모델 이름)를 자동으로 순회
    for name in models.keys():
        if name in summary[s]:
            m = summary[s][name]["mean"]
            sd = summary[s][name]["std"]
            print(f"  {name:9s}: {m:.2f} ± {sd:.2f} dB")

In [ ]:
random_summary = compute_random_scale_avg_psnr((2.0,3.0,4.0))

In [ ]:
# ============================================================
# 출력 2: 랜덤 스케일 평균 PSNR
# ============================================================
print("\n===== AVG Y-PSNR (Random scale per image) =====")
for name in models.keys():
    if name in random_summary:
        m = random_summary[name]["mean"]
        sd = random_summary[name]["std"]
        print(f"  {name:9s}: {m:.2f} ± {sd:.2f} dB")

In [ ]:
# ============================================================
# 8) 3종 모델 결과 시각화 및 드라이브 저장 (Visual Comparison)
# ============================================================
import os
import matplotlib.pyplot as plt

# 시각화 이미지가 저장될 드라이브 경로
SAVE_VIS_DIR = os.path.join(PROJECT_PATH, "results_comparison")
os.makedirs(SAVE_VIS_DIR, exist_ok=True)

@torch.no_grad()
def visualize_and_save_comparison(image_path, scale=4, crop_x=200, crop_y=200, crop_size=192):
    if not os.path.exists(image_path):
        print(f"⚠️ 이미지를 찾을 수 없습니다: {image_path}")
        return

    # 1. 원본 이미지 로드 및 자르기
    hr_img = Image.open(image_path).convert('RGB')
    w, h = hr_img.size

    # 예외 처리: 자르려는 위치가 이미지 크기를 벗어나면 조정
    crop_x = min(crop_x, max(0, w - crop_size))
    crop_y = min(crop_y, max(0, h - crop_size))

    hr_crop = hr_img.crop((crop_x, crop_y, crop_x + crop_size, crop_y + crop_size))

    # 텐서 변환 (Bicubic 통일)
    hr01 = transforms.ToTensor()(hr_crop).unsqueeze(0).to(DEVICE)
    lr01 = F.interpolate(hr01, size=(crop_size//scale, crop_size//scale), mode="bicubic", align_corners=False)

    # Bicubic (저해상도를 억지로 늘린 것)
    bicubic01 = F.interpolate(lr01, size=(crop_size, crop_size), mode="bicubic", align_corners=False)

    # 2. 각 모델별 추론 결과 담기
    results_dict = {}
    target_size = (crop_size, crop_size)

    for name, model in models.items():
        if "lno" in name.lower() or "upgrade" in name.lower():
            # LNO 계열 추론
            sr = model(lr01, target_size=target_size, scale_factor=float(scale))
            sr = torch.clamp(sr, 0.0, 1.0)
        else:
            # SRNO 계열 추론
            hr_in = norm_01_to_m11(hr01)
            lr_in = norm_01_to_m11(lr01)
            coords, cell = make_coord_grid(crop_size, crop_size, DEVICE, batch_size=1)
            cell = cell * torch.tensor(scale, device=DEVICE, dtype=torch.float32).view(1, 1)
            sr = model(lr_in, coords, cell)
            sr = sr.view(1, crop_size, crop_size, 3).permute(0,3,1,2).contiguous()
            sr = denorm_m11_to_01(sr)

        # PSNR 계산
        psnr = calc_psnr_y(sr, hr01, shave=2)

        # [핵심 수정] 텐서 메모리 덮어쓰기 방지를 위해 즉시 CPU numpy 배열로 분리 복사
        sr_np = sr.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
        results_dict[name] = {"image": sr_np, "psnr": psnr}

    # 3. 그림판(Figure) 세팅 및 시각화
    num_cols = 2 + len(models)
    fig, axes = plt.subplots(1, num_cols, figsize=(4 * num_cols, 4.5))

    file_name = os.path.basename(image_path)
    # 제목 폰트 스타일 수정 (볼드체 제거)
    fig.suptitle(f"{file_name} | Scale x{scale} | Crop({crop_x}, {crop_y})", fontsize=15)

    # (1) Bicubic
    bic_psnr = calc_psnr_y(bicubic01, hr01, shave=2)
    ax = axes[0]
    ax.imshow(bicubic01.squeeze(0).permute(1,2,0).cpu().numpy())
    ax.set_title(f"Bicubic\n{bic_psnr:.2f} dB", fontsize=12)
    ax.axis('off')

    # (2) Models
    for idx, (name, res) in enumerate(results_dict.items(), start=1):
        ax = axes[idx]
        ax.imshow(res["image"])
        ax.set_title(f"{name}\n{res['psnr']:.2f} dB", fontsize=12) # 볼드체(fontweight='bold') 제거
        ax.axis('off')

    # (3) Ground Truth (HR)
    ax = axes[-1]
    ax.imshow(hr01.squeeze(0).permute(1,2,0).cpu().numpy())
    ax.set_title("Ground Truth (HR)", fontsize=12) # PSNR: Inf 텍스트 제거
    ax.axis('off')

    plt.tight_layout()
    fig.subplots_adjust(top=0.8)

    # 4. 드라이브에 저장!
    save_path = os.path.join(SAVE_VIS_DIR, f"compare_{file_name}_x{scale}_x{crop_x}_y{crop_y}.png")
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

    # 터미널 출력
    print(f"📷 {file_name} (x{scale}, Crop: {crop_x},{crop_y}) PSNR 결과:")
    print(f"  - Bicubic: {bic_psnr:.2f} dB")
    for name, res in results_dict.items():
        print(f"  - {name}: {res['psnr']:.2f} dB")
    print(f"💾 이미지 저장 완료: {save_path}\n" + "-"*40)

# ============================================================
# 🚀 시각화 실행 테스트
# ============================================================
val_dir = os.path.join(LOCAL_DATA_PATH, 'DIV2K_valid_HR')

print("=== 🔍 3종 모델 시각화 및 비교 시작 ===")

# # (예시 1) 0891.png - x4 배율 비교
# visualize_and_save_comparison(os.path.join(val_dir, '0891.png'), scale=4, crop_x=507, crop_y=801, crop_size=192)

# # (예시 2) 0891.png - x3 배율 비교
# visualize_and_save_comparison(os.path.join(val_dir, '0891.png'), scale=3, crop_x=187, crop_y=801, crop_size=192)

# (예시 3) 0891.png - x2 배율 비교
visualize_and_save_comparison(os.path.join(val_dir, '0878.png'), scale=2, crop_x=504, crop_y=504, crop_size=192)

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

val_dir = os.path.join(LOCAL_DATA_PATH, 'DIV2K_valid_HR')
img_path = os.path.join(val_dir, '0801.png')

if os.path.exists(img_path):
    img = Image.open(img_path).convert('RGB')

    plt.figure(figsize=(15, 15))
    plt.imshow(img)
    plt.title("0801.png - Coordinate Finder", fontsize=16)

    # 100픽셀 단위로 촘촘하게 빨간 눈금선 그리기
    plt.xticks(range(0, img.width, 100), rotation=45)
    plt.yticks(range(0, img.height, 100))
    plt.grid(color='red', linestyle='--', linewidth=0.7, alpha=0.7)

    plt.show()
else:
    print("⚠️ 이미지를 찾을 수 없습니다.")

In [ ]:
# ============================================================
# 8) [분석 모드] 오차 히트맵(Error Map) & 복잡한 영역 자동 탐색
# ============================================================
import os
import matplotlib.pyplot as plt
import torch
import torchvision.transforms.functional as TF

SAVE_VIS_DIR = os.path.join(PROJECT_PATH, "results_comparison_error")
os.makedirs(SAVE_VIS_DIR, exist_ok=True)

def find_complex_crop(hr_img, crop_size=192, num_tries=50):
    """이미지 내에서 가장 픽셀 변화(Edge/Texture)가 심한 구역을 자동으로 찾습니다."""
    w, h = hr_img.size
    best_std = -1
    best_coords = (0, 0)

    # 여러 번 랜덤으로 찔러보고 가장 복잡한(표준편차가 큰) 곳을 선택
    for _ in range(num_tries):
        x = random.randint(0, max(0, w - crop_size))
        y = random.randint(0, max(0, h - crop_size))
        crop = hr_img.crop((x, y, x + crop_size, y + crop_size))

        # 흑백 변환 후 표준편차(복잡도) 계산
        std = transforms.ToTensor()(crop.convert('L')).std().item()
        if std > best_std:
            best_std = std
            best_coords = (x, y)

    return best_coords

@torch.no_grad()
def visualize_with_errormap(image_path, scale=4, crop_size=192):
    if not os.path.exists(image_path):
        return

    # 1. 원본 이미지 로드 및 가장 복잡한 영역 자동 탐색
    hr_img = Image.open(image_path).convert('RGB')
    crop_x, crop_y = find_complex_crop(hr_img, crop_size)

    hr_crop = hr_img.crop((crop_x, crop_y, crop_x + crop_size, crop_y + crop_size))

    # 텐서 변환 (Bicubic 통일)
    hr01 = transforms.ToTensor()(hr_crop).unsqueeze(0).to(DEVICE)
    lr01 = F.interpolate(hr01, size=(crop_size//scale, crop_size//scale), mode="bicubic", align_corners=False)
    bicubic01 = F.interpolate(lr01, size=(crop_size, crop_size), mode="bicubic", align_corners=False)

    # 2. 각 모델별 추론 및 오차맵 계산
    results_dict = {}
    target_size = (crop_size, crop_size)

    # Bicubic 결과 먼저 추가
    bic_psnr = calc_psnr_y(bicubic01, hr01, shave=2)
    bic_err = torch.abs(bicubic01 - hr01).mean(dim=1).squeeze(0).cpu().numpy() # 채널 평균 절대 오차
    results_dict["Bicubic"] = {"image": bicubic01.squeeze(0).permute(1,2,0).cpu().numpy(), "psnr": bic_psnr, "err": bic_err}

    for name, model in models.items():
        if "lno" in name.lower() or "upgrade" in name.lower():
            sr = model(lr01, target_size=target_size, scale_factor=float(scale))
            sr = torch.clamp(sr, 0.0, 1.0)
        else:
            hr_in = norm_01_to_m11(hr01)
            lr_in = norm_01_to_m11(lr01)
            coords, cell = make_coord_grid(crop_size, crop_size, DEVICE, batch_size=1)
            cell = cell * torch.tensor(scale, device=DEVICE, dtype=torch.float32).view(1, 1)
            sr = model(lr_in, coords, cell)
            sr = sr.view(1, crop_size, crop_size, 3).permute(0,3,1,2).contiguous()
            sr = denorm_m11_to_01(sr)

        psnr = calc_psnr_y(sr, hr01, shave=2)
        sr_np = sr.squeeze(0).permute(1, 2, 0).detach().cpu().numpy().copy()

        # [핵심] 정답과의 절대 오차(Absolute Error) 맵 생성
        err_map = torch.abs(sr - hr01).mean(dim=1).squeeze(0).cpu().numpy()
        results_dict[name] = {"image": sr_np, "psnr": psnr, "err": err_map}

    # 3. 그림판 세팅 (위줄: 원본 이미지, 아랫줄: 오차 히트맵)
    num_cols = len(results_dict) + 1 # 모델 수 + Bicubic + 정답
    fig, axes = plt.subplots(2, num_cols, figsize=(3.5 * num_cols, 7))

    file_name = os.path.basename(image_path)
    fig.suptitle(f"{file_name} | Scale x{scale} | Auto-Complex Crop({crop_x}, {crop_y})", fontsize=15)

    # (1) 예측 이미지들 & 오차 히트맵
    vmax = 0.2 # 에러맵 최대 색상 임계치 (빨간색이 진해지는 기준)

    for idx, (name, res) in enumerate(results_dict.items()):
        # 위쪽 행: 복원 이미지
        axes[0, idx].imshow(res["image"])
        axes[0, idx].set_title(f"{name}\n{res['psnr']:.2f} dB", fontsize=12)
        axes[0, idx].axis('off')

        # 아래쪽 행: 오차 히트맵 (에러가 클수록 붉은색)
        im = axes[1, idx].imshow(res["err"], cmap='jet', vmin=0, vmax=vmax)
        axes[1, idx].set_title(f"{name} Error Map", fontsize=10)
        axes[1, idx].axis('off')

    # (2) Ground Truth
    axes[0, -1].imshow(hr01.squeeze(0).permute(1,2,0).cpu().numpy())
    axes[0, -1].set_title("Ground Truth (HR)", fontsize=12)
    axes[0, -1].axis('off')

    axes[1, -1].axis('off') # GT의 에러맵 자리는 비워둠

    plt.tight_layout()
    fig.subplots_adjust(top=0.88)

    # 4. 저장
    save_path = os.path.join(SAVE_VIS_DIR, f"errormap_{file_name}_x{scale}_x{crop_x}_y{crop_y}.png")
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# ============================================================
# 🚀 오차 히트맵 분석 실행
# ============================================================
val_dir = os.path.join(LOCAL_DATA_PATH, 'DIV2K_valid_HR')

print("=== 🔍 복잡한 영역 자동 탐색 & 오차 히트맵 시각화 ===")
# 알아서 가장 복잡한 곳을 찾아서 시각화해 줍니다!
visualize_with_errormap(os.path.join(val_dir, '0891.png'), scale=4, crop_size=192)